## 3. Exemplos de IPyWidgets com Dicionários em Cascata

Aqui estão alguns exemplos práticos de como usar ipywidgets para navegar em dicionários aninhados de forma interativa.

### Exemplo 1: Dropdown em Cascata Simples

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Dicionário em cascata de exemplo
dados_estudo = {
    "Idiomas": {
        "Inglês": {
            "Gramática": ["Present Perfect", "Past Continuous", "Future Simple"],
            "Vocabulário": ["Business", "Academic", "Daily Life"],
            "Listening": ["Ted Talks", "Podcasts", "Movies"]
        },
        "Espanhol": {
            "Gramática": ["Subjuntivo", "Pretérito", "Futuro"],
            "Vocabulário": ["Formal", "Informal", "Técnico"],
            "Conversação": ["Situações cotidianas", "Negócios", "Viagens"]
        }
    },
    "Programação": {
        "Python": {
            "Básico": ["Variáveis", "Loops", "Funções"],
            "Intermediário": ["Classes", "Decorators", "Generators"],
            "Avançado": ["Metaclasses", "Async/Await", "Performance"]
        },
        "JavaScript": {
            "Frontend": ["DOM", "Events", "Fetch API"],
            "Backend": ["Node.js", "Express", "APIs"],
            "Frameworks": ["React", "Vue", "Angular"]
        }
    }
}

# Widgets
categoria_dropdown = widgets.Dropdown(
    options=list(dados_estudo.keys()),
    description='Categoria:'
)

subcategoria_dropdown = widgets.Dropdown(
    description='Subcategoria:'
)

topico_dropdown = widgets.Dropdown(
    description='Tópico:'
)

conteudo_dropdown = widgets.Dropdown(
    description='Conteúdo:'
)

output_area = widgets.Output()

def atualizar_subcategoria(change):
    with output_area:
        clear_output()
    categoria_selecionada = change['new']
    subcategorias = list(dados_estudo[categoria_selecionada].keys())
    subcategoria_dropdown.options = subcategorias
    subcategoria_dropdown.value = subcategorias[0] if subcategorias else None

def atualizar_topico(change):
    with output_area:
        clear_output()
    categoria = categoria_dropdown.value
    subcategoria = change['new']
    topicos = list(dados_estudo[categoria][subcategoria].keys())
    topico_dropdown.options = topicos
    topico_dropdown.value = topicos[0] if topicos else None

def atualizar_conteudo(change):
    with output_area:
        clear_output()
    categoria = categoria_dropdown.value
    subcategoria = subcategoria_dropdown.value
    topico = change['new']
    conteudos = dados_estudo[categoria][subcategoria][topico]
    conteudo_dropdown.options = conteudos
    conteudo_dropdown.value = conteudos[0] if conteudos else None

def mostrar_selecao(change):
    with output_area:
        clear_output()
        print(f"Seleção atual:")
        print(f"Categoria: {categoria_dropdown.value}")
        print(f"Subcategoria: {subcategoria_dropdown.value}")
        print(f"Tópico: {topico_dropdown.value}")
        print(f"Conteúdo: {change['new']}")
        print(f"\nCaminho completo: {categoria_dropdown.value} → {subcategoria_dropdown.value} → {topico_dropdown.value} → {change['new']}")

# Conectar eventos
categoria_dropdown.observe(atualizar_subcategoria, names='value')
subcategoria_dropdown.observe(atualizar_topico, names='value')
topico_dropdown.observe(atualizar_conteudo, names='value')
conteudo_dropdown.observe(mostrar_selecao, names='value')

# Inicializar
atualizar_subcategoria({'new': categoria_dropdown.value})

# Layout
ui = widgets.VBox([
    categoria_dropdown,
    subcategoria_dropdown,
    topico_dropdown,
    conteudo_dropdown,
    output_area
])

display(ui)

### Exemplo 2: Sistema Mais Avançado com Classe

In [ ]:
class NavigadorDicionario:
    def __init__(self, dados):
        self.dados = dados
        self.caminho_atual = []
        self.criar_widgets()
        self.conectar_eventos()
        
    def criar_widgets(self):
        self.widgets_navegacao = []
        self.output = widgets.Output()
        
        # Criar primeiro nível
        primeiro_widget = widgets.Dropdown(
            options=list(self.dados.keys()),
            description=f'Nível 1:'
        )
        self.widgets_navegacao.append(primeiro_widget)
        
    def conectar_eventos(self):
        for i, widget in enumerate(self.widgets_navegacao):
            widget.observe(lambda change, nivel=i: self.atualizar_nivel(change, nivel), names='value')
    
    def atualizar_nivel(self, change, nivel):
        # Limpar widgets dos níveis seguintes
        while len(self.widgets_navegacao) > nivel + 1:
            self.widgets_navegacao.pop()
        
        # Atualizar caminho atual
        self.caminho_atual = [w.value for w in self.widgets_navegacao[:nivel+1]]
        
        # Obter dados atuais
        dados_atuais = self.dados
        for chave in self.caminho_atual:
            dados_atuais = dados_atuais[chave]
        
        # Se ainda há mais níveis para navegar
        if isinstance(dados_atuais, dict) and dados_atuais:
            proximo_widget = widgets.Dropdown(
                options=list(dados_atuais.keys()),
                description=f'Nível {nivel + 2}:'
            )
            self.widgets_navegacao.append(proximo_widget)
            proximo_widget.observe(
                lambda change, nivel=nivel+1: self.atualizar_nivel(change, nivel), 
                names='value'
            )
        
        self.mostrar_resultado()
        self.atualizar_display()
    
    def mostrar_resultado(self):
        with self.output:
            clear_output()
            print("Caminho navegado:", " → ".join(self.caminho_atual))
            
            # Obter valor atual
            valor_atual = self.dados
            for chave in self.caminho_atual:
                valor_atual = valor_atual[chave]
            
            if isinstance(valor_atual, dict):
                print(f"Opções disponíveis: {list(valor_atual.keys())}")
            elif isinstance(valor_atual, list):
                print(f"Lista encontrada com {len(valor_atual)} itens:")
                for i, item in enumerate(valor_atual, 1):
                    print(f"  {i}. {item}")
            else:
                print(f"Valor final: {valor_atual}")
    
    def atualizar_display(self):
        self.ui = widgets.VBox(self.widgets_navegacao + [self.output])
        
    def display(self):
        self.atualizar_display()
        display(self.ui)

# Dados de exemplo mais complexos
dados_anki = {
    "Matérias": {
        "Idiomas": {
            "Inglês": {
                "Básico": {
                    "Verbos": ["to be", "to have", "to do"],
                    "Substantivos": ["house", "car", "book"],
                    "Adjetivos": ["big", "small", "beautiful"]
                },
                "Intermediário": {
                    "Phrasal Verbs": ["get up", "turn on", "look for"],
                    "Expressões": ["How are you?", "What's up?", "See you later"]
                }
            },
            "Espanhol": {
                "Básico": {
                    "Verbos": ["ser", "estar", "tener"],
                    "Substantivos": ["casa", "coche", "libro"]
                }
            }
        },
        "Matemática": {
            "Álgebra": {
                "Equações": ["Primeiro grau", "Segundo grau", "Sistemas"],
                "Funções": ["Linear", "Quadrática", "Exponencial"]
            },
            "Geometria": {
                "Plana": ["Triângulos", "Círculos", "Polígonos"],
                "Espacial": ["Prismas", "Pirâmides", "Esferas"]
            }
        }
    },
    "Configurações": {
        "Dificuldade": ["Fácil", "Médio", "Difícil"],
        "Intervalos": {
            "Inicial": 1,
            "Fácil": 4,
            "Médio": 2,
            "Difícil": 1
        }
    }
}

# Criar e exibir o navegador
navegador = NavigadorDicionario(dados_anki)
navegador.display()

### Exemplo 3: Sistema com Busca e Filtros

In [ ]:
class ExploradorAvancado:
    def __init__(self, dados):
        self.dados = dados
        self.dados_filtrados = dados
        self.criar_interface()
        
    def criar_interface(self):
        # Widget de busca
        self.busca_text = widgets.Text(
            placeholder='Digite para buscar...',
            description='Buscar:'
        )
        
        # Botão para resetar busca
        self.reset_button = widgets.Button(
            description='Resetar',
            button_style='warning'
        )
        
        # Seletor de categoria
        self.categoria_select = widgets.SelectMultiple(
            options=[],
            description='Categorias:',
            rows=4
        )
        
        # Output para resultados
        self.output = widgets.Output()
        
        # Output para detalhes
        self.detalhes_output = widgets.Output()
        
        # Conectar eventos
        self.busca_text.observe(self.buscar, names='value')
        self.reset_button.on_click(self.resetar)
        self.categoria_select.observe(self.mostrar_detalhes, names='value')
        
        # Layout inicial
        self.atualizar_categorias()
        
    def buscar(self, change):
        termo = change['new'].lower()
        
        if not termo:
            self.dados_filtrados = self.dados
        else:
            self.dados_filtrados = self.filtrar_recursivo(self.dados, termo)
        
        self.atualizar_categorias()
        self.mostrar_resultados_busca(termo)
    
    def filtrar_recursivo(self, dados, termo):
        resultado = {}
        
        for chave, valor in dados.items():
            if isinstance(valor, dict):
                sub_resultado = self.filtrar_recursivo(valor, termo)
                if sub_resultado or termo in chave.lower():
                    resultado[chave] = sub_resultado if sub_resultado else valor
            elif isinstance(valor, list):
                itens_filtrados = [item for item in valor if termo in str(item).lower()]
                if itens_filtrados or termo in chave.lower():
                    resultado[chave] = itens_filtrados if itens_filtrados else valor
            else:
                if termo in str(valor).lower() or termo in chave.lower():
                    resultado[chave] = valor
        
        return resultado
    
    def atualizar_categorias(self):
        opcoes = self.extrair_caminhos(self.dados_filtrados)
        self.categoria_select.options = opcoes
    
    def extrair_caminhos(self, dados, caminho_atual=""):
        caminhos = []
        
        for chave, valor in dados.items():
            novo_caminho = f"{caminho_atual} → {chave}" if caminho_atual else chave
            
            if isinstance(valor, dict):
                caminhos.append(novo_caminho)
                caminhos.extend(self.extrair_caminhos(valor, novo_caminho))
            else:
                caminhos.append(novo_caminho)
        
        return sorted(caminhos)
    
    def mostrar_resultados_busca(self, termo):
        with self.output:
            clear_output()
            if termo:
                print(f"Resultados para '{termo}':")
                print(f"Encontradas {len(self.categoria_select.options)} categorias")
            else:
                print("Todas as categorias disponíveis")
    
    def mostrar_detalhes(self, change):
        with self.detalhes_output:
            clear_output()
            
            if not change['new']:
                return
                
            for caminho_selecionado in change['new']:
                print(f"\\n=== {caminho_selecionado} ===")
                
                # Navegar até o valor
                partes = [p.strip() for p in caminho_selecionado.split('→')]
                valor_atual = self.dados_filtrados
                
                try:
                    for parte in partes:
                        valor_atual = valor_atual[parte]
                    
                    if isinstance(valor_atual, list):
                        print(f"Lista com {len(valor_atual)} itens:")
                        for i, item in enumerate(valor_atual, 1):
                            print(f"  {i}. {item}")
                    elif isinstance(valor_atual, dict):
                        print(f"Dicionário com {len(valor_atual)} chaves:")
                        for chave in valor_atual.keys():
                            print(f"  - {chave}")
                    else:
                        print(f"Valor: {valor_atual}")
                        
                except KeyError as e:
                    print(f"Erro ao acessar: {e}")
    
    def resetar(self, button):
        self.busca_text.value = ""
        self.dados_filtrados = self.dados
        self.atualizar_categorias()
        with self.output:
            clear_output()
        with self.detalhes_output:
            clear_output()
    
    def display(self):
        # Layout da interface
        busca_box = widgets.HBox([self.busca_text, self.reset_button])
        
        main_box = widgets.HBox([
            widgets.VBox([
                widgets.HTML("<b>Navegação:</b>"),
                self.categoria_select
            ]),
            widgets.VBox([
                widgets.HTML("<b>Detalhes:</b>"),
                self.detalhes_output
            ])
        ])
        
        ui = widgets.VBox([
            busca_box,
            self.output,
            main_box
        ])
        
        display(ui)

# Dados para o exemplo
biblioteca_conhecimento = {
    "Programação": {
        "Python": {
            "Web": {
                "Django": ["Models", "Views", "Templates", "URLs"],
                "Flask": ["Routes", "Templates", "Blueprints"],
                "FastAPI": ["Async", "Pydantic", "OpenAPI"]
            },
            "Data Science": {
                "Pandas": ["DataFrames", "Series", "Groupby", "Merge"],
                "NumPy": ["Arrays", "Broadcasting", "Linear Algebra"],
                "Matplotlib": ["Plots", "Subplots", "Styling"]
            },
            "Automação": ["Selenium", "BeautifulSoup", "Requests"]
        },
        "JavaScript": {
            "Frontend": {
                "React": ["Components", "Hooks", "State", "Props"],
                "Vue": ["Directives", "Components", "Vuex"],
                "Vanilla": ["DOM", "Events", "Fetch", "Promises"]
            },
            "Backend": {
                "Node.js": ["Express", "Middleware", "Routing"],
                "APIs": ["REST", "GraphQL", "WebSockets"]
            }
        }
    },
    "Idiomas": {
        "Inglês": {
            "Gramática": ["Tenses", "Conditionals", "Modal Verbs"],
            "Vocabulário": {
                "Business": ["Meetings", "Presentations", "Negotiations"],
                "Academic": ["Research", "Writing", "Citations"],
                "Daily": ["Shopping", "Travel", "Weather"]
            }
        },
        "Espanhol": {
            "Gramática": ["Subjuntivo", "Ser vs Estar", "Por vs Para"],
            "Cultura": ["Países", "Tradições", "Literatura"]
        }
    }
}

# Criar e exibir o explorador
explorador = ExploradorAvancado(biblioteca_conhecimento)
explorador.display()

### Exemplo 4: Sistema Específico para seu Projeto Anki

In [ ]:
class GerenciadorCartoes:
    def __init__(self):
        # Estrutura de dados para organizar os cartões
        self.estrutura_cartoes = {
            "020000": {  # Idiomas
                "nome": "Idiomas",
                "subcategorias": {
                    "020100": {  # Inglês
                        "nome": "Inglês",
                        "grupos": {
                            "020101": {  # Ted Talks
                                "nome": "Ted Talks",
                                "configuracao": {
                                    "c1": 0.8,
                                    "c2": 1.0,
                                    "formula": "(c1 * 2) ** (x * c2)"
                                },
                                "cartoes": []
                            },
                            "020102": {
                                "nome": "Gramática Básica",
                                "configuracao": {
                                    "c1": 0.9,
                                    "c2": 0.8,
                                    "formula": "(c1 * 2) ** (x * c2)"
                                },
                                "cartoes": []
                            }
                        }
                    },
                    "020200": {  # Espanhol
                        "nome": "Espanhol",
                        "grupos": {
                            "020201": {
                                "nome": "Conversação",
                                "configuracao": {
                                    "c1": 0.7,
                                    "c2": 1.1,
                                    "formula": "(c1 * 2) ** (x * c2)"
                                },
                                "cartoes": []
                            }
                        }
                    }
                }
            },
            "030000": {  # Programação
                "nome": "Programação",
                "subcategorias": {
                    "030100": {
                        "nome": "Python",
                        "grupos": {
                            "030101": {
                                "nome": "Data Science",
                                "configuracao": {
                                    "c1": 0.85,
                                    "c2": 0.9,
                                    "formula": "(c1 * 2) ** (x * c2)"
                                },
                                "cartoes": []
                            }
                        }
                    }
                }
            }
        }
        
        self.criar_interface()
    
    def criar_interface(self):
        # Widgets de seleção
        self.categoria_widget = widgets.Dropdown(
            options=[(dados["nome"], id_cat) for id_cat, dados in self.estrutura_cartoes.items()],
            description='Categoria:'
        )
        
        self.subcategoria_widget = widgets.Dropdown(
            description='Subcategoria:'
        )
        
        self.grupo_widget = widgets.Dropdown(
            description='Grupo:'
        )
        
        # Widgets de configuração
        self.c1_widget = widgets.FloatSlider(
            value=0.8,
            min=0.1,
            max=2.0,
            step=0.1,
            description='C1:'
        )
        
        self.c2_widget = widgets.FloatSlider(
            value=1.0,
            min=0.1,
            max=2.0,
            step=0.1,
            description='C2:'
        )
        
        # Widget para adicionar cartão
        self.novo_cartao_widget = widgets.Text(
            placeholder='Digite o conteúdo do novo cartão...',
            description='Novo cartão:'
        )
        
        self.adicionar_button = widgets.Button(
            description='Adicionar Cartão',
            button_style='success'
        )
        
        # Output areas
        self.info_output = widgets.Output()
        self.grafico_output = widgets.Output()
        
        # Conectar eventos
        self.categoria_widget.observe(self.atualizar_subcategorias, names='value')
        self.subcategoria_widget.observe(self.atualizar_grupos, names='value')
        self.grupo_widget.observe(self.carregar_configuracao, names='value')
        self.c1_widget.observe(self.atualizar_grafico, names='value')
        self.c2_widget.observe(self.atualizar_grafico, names='value')
        self.adicionar_button.on_click(self.adicionar_cartao)
        
        # Inicializar
        self.atualizar_subcategorias({'new': self.categoria_widget.value})
    
    def atualizar_subcategorias(self, change):
        categoria_id = change['new']
        subcategorias = self.estrutura_cartoes[categoria_id]['subcategorias']
        
        self.subcategoria_widget.options = [
            (dados["nome"], id_sub) for id_sub, dados in subcategorias.items()
        ]
        
        if subcategorias:
            self.subcategoria_widget.value = list(subcategorias.keys())[0]
    
    def atualizar_grupos(self, change):
        categoria_id = self.categoria_widget.value
        subcategoria_id = change['new']
        grupos = self.estrutura_cartoes[categoria_id]['subcategorias'][subcategoria_id]['grupos']
        
        self.grupo_widget.options = [
            (dados["nome"], id_grupo) for id_grupo, dados in grupos.items()
        ]
        
        if grupos:
            self.grupo_widget.value = list(grupos.keys())[0]
    
    def carregar_configuracao(self, change):
        grupo_info = self.obter_grupo_atual()
        if grupo_info:
            config = grupo_info['configuracao']
            self.c1_widget.value = config['c1']
            self.c2_widget.value = config['c2']
            
        self.atualizar_info()
        self.atualizar_grafico()
    
    def obter_grupo_atual(self):
        try:
            categoria_id = self.categoria_widget.value
            subcategoria_id = self.subcategoria_widget.value
            grupo_id = self.grupo_widget.value
            
            return self.estrutura_cartoes[categoria_id]['subcategorias'][subcategoria_id]['grupos'][grupo_id]
        except:
            return None
    
    def atualizar_info(self, change=None):
        with self.info_output:
            clear_output()
            
            grupo_info = self.obter_grupo_atual()
            if grupo_info:
                print(f"Grupo selecionado: {grupo_info['nome']}")
                print(f"ID completo: {self.grupo_widget.value}")
                print(f"Fórmula: {grupo_info['configuracao']['formula']}")
                print(f"Parâmetros: C1={grupo_info['configuracao']['c1']}, C2={grupo_info['configuracao']['c2']}")
                print(f"Cartões cadastrados: {len(grupo_info['cartoes'])}")
                
                if grupo_info['cartoes']:
                    print("\\nCartões existentes:")
                    for i, cartao in enumerate(grupo_info['cartoes'], 1):
                        print(f"  {i}. {cartao}")
    
    def atualizar_grafico(self, change=None):
        with self.grafico_output:
            clear_output()
            
            c1 = self.c1_widget.value
            c2 = self.c2_widget.value
            
            x = np.arange(0, 15, 1)
            y = (c1 * 2) ** (x * c2)
            
            plt.figure(figsize=(10, 6))
            plt.plot(x, y, marker='o', linewidth=2, markersize=6)
            plt.xlabel('Nível do Cartão')
            plt.ylabel('Dias até Próxima Revisão')
            plt.title(f'Curva de Revisão: (C1 * 2) ^ (x * C2)\\nC1 = {c1}, C2 = {c2}')
            plt.grid(True, alpha=0.3)
            plt.xlim(left=0)
            plt.ylim(bottom=0)
            
            # Adicionar valores nas barras
            for i, v in enumerate(y):
                if i % 2 == 0:  # Mostrar apenas valores pares para não poluir
                    plt.annotate(f'{v:.1f}', (i, v), textcoords="offset points", 
                               xytext=(0,10), ha='center', fontsize=8)
            
            plt.tight_layout()
            plt.show()
    
    def adicionar_cartao(self, button):
        conteudo = self.novo_cartao_widget.value.strip()
        if not conteudo:
            return
        
        grupo_info = self.obter_grupo_atual()
        if grupo_info:
            grupo_info['cartoes'].append(conteudo)
            self.novo_cartao_widget.value = ""
            self.atualizar_info()
            
            with self.info_output:
                print(f"\\n✅ Cartão adicionado com sucesso!")
    
    def display(self):
        # Layout da interface
        selecao_box = widgets.VBox([
            widgets.HTML("<h3>Seleção de Categoria</h3>"),
            self.categoria_widget,
            self.subcategoria_widget,
            self.grupo_widget
        ])
        
        config_box = widgets.VBox([
            widgets.HTML("<h3>Configuração</h3>"),
            self.c1_widget,
            self.c2_widget,
            self.novo_cartao_widget,
            self.adicionar_button
        ])
        
        info_box = widgets.VBox([
            widgets.HTML("<h3>Informações</h3>"),
            self.info_output
        ])
        
        superior = widgets.HBox([selecao_box, config_box, info_box])
        
        ui = widgets.VBox([
            superior,
            widgets.HTML("<h3>Curva de Revisão</h3>"),
            self.grafico_output
        ])
        
        display(ui)
        
        # Carregar dados iniciais
        self.carregar_configuracao({'new': None})

# Criar e exibir o gerenciador
gerenciador = GerenciadorCartoes()
gerenciador.display()

## Resumo dos Exemplos

### Exemplo 1: Dropdown em Cascata Simples
- Demonstra como criar dropdowns que se atualizam em cascata
- Ideal para navegação linear em estruturas hierárquicas
- Mostra como conectar eventos entre widgets

### Exemplo 2: Sistema com Classe (NavigadorDicionario)
- Versão mais robusta e reutilizável
- Cria widgets dinamicamente conforme a estrutura dos dados
- Suporta qualquer profundidade de dicionários aninhados

### Exemplo 3: Sistema com Busca (ExploradorAvançado)
- Adiciona funcionalidade de busca e filtros
- Permite busca em texto livre através da estrutura
- Interface mais rica com múltiplas áreas de exibição

### Exemplo 4: Sistema Específico para Anki (GerenciadorCartoes)
- Implementação customizada para seu projeto
- Integra navegação com configuração de parâmetros
- Inclui visualização gráfica e adição de cartões
- Estrutura de dados específica para sistema de spaced repetition

Cada exemplo aumenta em complexidade e funcionalidade, permitindo que você escolha a abordagem mais adequada para suas necessidades específicas.